# ST554_Final Project

Final project for ST554: Analysis of Big Data at NC State University, Spring 2026

Creator: Cole Hammett

Purpose: Predicting power consumption (Zone 3) for Tetouan City using PySpark MLlib and Structured Streaming.

Date: 30th of April, 2026

Instructor: Justin Post

# Part 1: Fitting Your Model

# 1. Setup & Data Loading

In [1]:
# Import necessary modules

import pandas as pd
from pyspark.sql import SparkSession

## 1.1 Starting a SparkSession

`local[*]` tells Spark to run locally using all available CPU cores.

`appName` is just a label for the UI/logs.

In [2]:
# Create a SparkSession using all available local cores
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('power_zone_prediction') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 15:40:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 15:40:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/30 15:40:32 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/30 15:40:32 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/30 15:40:32 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/04/30 15:40:32 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/04/30 15:40:32 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.


## 1.2 Reading data with pandas

Two-step load, read in .csv with pandas THEN convert rather than reading directly into Spark

In [3]:
# Read power_ml_data.csv directly from the course URL into a pandas DataFrame
power_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")

In [4]:
# Quick sanity checks on the pandas DataFrame
print("Shape:", power_pd.shape)
print("\nFirst few rows:")
power_pd.head()

Shape: (47174, 10)

First few rows:


,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


## 1.3 Converting to a Spark Dataframe

spark.createDataFrame() accepts a pandas DataFrame directly

In [5]:
spark_df = spark.createDataFrame(power_pd)

### 1.3.1 Verifying the schema for catching type issues before pipeline work

In particular, checked whether Hour is already a DoubleType rather than `LongType` or `IntegerType`

In [6]:
print("Spark DataFrame schema:")
spark_df.printSchema()

Spark DataFrame schema:
root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



### 1.3.2 Previewing the first few rows of the Spark DataFrame

In [7]:
spark_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## 1.4 Summarizing data with Spark

Used Spark's built-in aggregation tools to understand the structure, distributions, and relationships in the data before building the pipeline. This informs modeling choices
(e.g., why PCA on weather columns, why binarize Hour).

### 1.4.1 Total number of observations and features


In [ ]:
print(f"Rows:    {spark_df.count():,}")
print(f"Columns: {len(spark_df.columns)}")
print(f"Columns: {spark_df.columns}")

### 1.4.2 Descriptive statistics on numeric columns

`.describe()` called Spark's built-in summary for all numeric columns, returning count, mean, standard deviation, min, and max. This is the quickest way to flag outliers or scale differences across predictors before fitting.

In [ ]:
spark_df.describe(
    "Temperature", "Humidity", "Wind_Speed",
    "General_Diffuse_Flows", "Diffuse_Flows",
    "Power_Zone_1", "Power_Zone_2", "Power_Zone_3"
).show()

### 1.4.3 Missing value check

Structured streaming and MLlib pipelines do not handle nulls gracefully; a single null in a feature column will propagate NaN predictions. 

Confirmed there are no missing values before building the pipeline.

In [ ]:
from pyspark.sql.functions import col, isnan, when, count

null_counts = spark_df.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    for c in spark_df.columns
])

print("Null / NaN counts per column:")
null_counts.show()

### 1.4.4 Observation Counts and Mean Power_Zone_3 by Month

Grouping by Month revealed both seasonal patterns in the response variable and whether observations are evenly distributed across months; an imbalanced month could affect
cross-validation fold quality.

In [ ]:
from pyspark.sql.functions import avg, round as spark_round

# Observations per month and average response (Power_Zone_3) per month
(spark_df
    .groupBy("Month")
    .agg(
        count("*").alias("n_obs"),
        spark_round(avg("Power_Zone_3"), 2).alias("avg_Power_Zone_3"),
        spark_round(avg("Temperature"), 2).alias("avg_Temperature"),
        spark_round(avg("Humidity"), 2).alias("avg_Humidity")
    )
    .orderBy("Month")
    .show()
)

### 1.4.5 Observation Counts by Hour

Hour ranges 0–23. Binarized it at 6.5 (night = Hour ≤ 6, day = Hour ≥ 7).
This summary confirmed how many observations fall into each bin and whether the
threshold makes intuitive sense given the average response.

In [ ]:
from pyspark.sql.functions import when

(spark_df
    .withColumn(
        "Hour_bin",
        when(col("Hour") <= 6, "night (0)").otherwise("day (1)")
    )
    .groupBy("Hour", "Hour_bin")
    .agg(
        count("*").alias("n_obs"),
        spark_round(avg("Power_Zone_3"), 2).alias("avg_Power_Zone_3")
    )
    .orderBy("Hour")
    .show(24)
)

### 1.4.6 Pairwise Correlations with Power_Zone_3

Spark's `.stat.corr()` computed Pearson correlation between two columns. Running it for each predictor against the response helps confirm which variables carry strong linear signal and which (like weather columns) may benefit from dimensionality reduction via PCA.

In [ ]:
numeric_predictors = [
    "Temperature", "Humidity", "Wind_Speed",
    "General_Diffuse_Flows", "Diffuse_Flows",
    "Power_Zone_1", "Power_Zone_2"
]

print(f"{'Predictor':<25} {'Corr with Power_Zone_3':>22}")
print("-" * 50)
for col_name in numeric_predictors:
    r = spark_df.stat.corr(col_name, "Power_Zone_3")
    print(f"{col_name:<25} {r:>22.4f}")

### 1.4.7 Night vs. Day Summary

A high-level split on the binarized Hour variable shows whether power consumption
differs meaningfully between night (Hour ≤ 6) and day (Hour > 6). This validates
the modeling decision to include the binary flag as a predictor.

In [ ]:
(spark_df
    .withColumn(
        "time_of_day",
        when(col("Hour") <= 6, "Night (Hour <= 6)").otherwise("Day (Hour > 6)")
    )
    .groupBy("time_of_day")
    .agg(
        count("*").alias("n_obs"),
        spark_round(avg("Power_Zone_3"), 2).alias("avg_Power_Zone_3"),
        spark_round(avg("Power_Zone_1"), 2).alias("avg_Power_Zone_1"),
        spark_round(avg("Power_Zone_2"), 2).alias("avg_Power_Zone_2"),
        spark_round(avg("Temperature"),  2).alias("avg_Temperature")
    )
    .orderBy("time_of_day")
    .show()
)

# 2. Transformer Pipeline

Each sub-step below becomes a stage in a `Pipeline()`. The one exception is PCA, which must be pre-fit outside the pipeline so that a fitted transformer (not an estimator) is passed in as a stage.

In [8]:
# Import all MLlib components needed for the transformation pipeline
from pyspark.ml import Pipeline
from pyspark.ml.feature import (SQLTransformer, Binarizer, OneHotEncoder,
                                 VectorAssembler, PCA)

## 2.1 Casting Hour to DoubleType

 - The schema check above showed Hour is `LongType`. 
 - `VectorAssembler` requires `DoubleType`.
 - SQLTransformer rewrites the whole table in-place using `__THIS__` as the table alias.
 - Casted Hour to DoubleType under a new name to avoid a duplicate-column conflict.
 - `SELECT *` keeps the original `LongType` Hour; the cast result goes into `Hour_d`.
 
Note: `sql_cast` uses `SELECT *, CAST(...)` rather than listing every column which keeps all original columns intact so downstream stages can still reference them.

In [9]:
sql_cast = SQLTransformer(
    statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_d FROM __THIS__"
)

## 2.2 Binarizing `Hour`

- Binarized Hour: 0 if Hour <= 6.5 (night), 1 if Hour > 6.5 (day).
- inputCol read the freshly-cast DoubleType Hour produced by sql_cast above.

In [10]:
binarizer = Binarizer(
    threshold=6.5,
    inputCol="Hour_d",
    outputCol="Hour_bin"
)

## 2.3 One-Hot Encoding `Month`

Month is already numeric (1–12), so no StringIndexer is needed. OHE will produce a sparse vector of length 11 (one category dropped by default).

In [11]:
ohe = OneHotEncoder(
    inputCol="Month",
    outputCol="Month_ohe"
)

## 2.4 Pre-fitting PCA on weather columns

The trickiest part. `pca_estimator.fit()` returns a `PCAModel` (transformer); that's what goes into the pipeline, not `pca_estimator` itself.

### 2.4.1 Assembling the five weather columns into a single vector column

`weather_assembler` appears as a pipeline stage so it runs on streaming data later. Cannot just rely on the `weather_vec_df` created during the pre-fit.

In [12]:
weather_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed",
               "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="weather_vec"
)

### 2.4.2 Fitting the PCA estimator on the assembled weather vectors

Fitted here (outside the pipeline) so that the *fitted* PCA model(a transformer) is passed into the pipeline rather than the unfitted estimator.

In [13]:
pca_estimator = PCA(k=2, inputCol="weather_vec", outputCol="pca_features")

weather_vec_df = weather_assembler.transform(spark_df)
pca_model = pca_estimator.fit(weather_vec_df)

print("PCA explained variance ratios:", pca_model.explainedVariance)

PCA explained variance ratios: [0.883862615096907,0.11355803317669594]


## 2.5 Renaming response as `label`

Renamed Power_Zone_3 to label as required by MLlib's LinearRegression. SELECT * preserved all existing columns while adding the alias.

In [14]:
sql_label = SQLTransformer(
    statement="SELECT *, Power_Zone_3 AS label FROM __THIS__"
)

## 2.6 Combining all predictors into a single 'features' vector

- 2 PCA components from the weather columns
- binary Hour flag (night vs. day)
- Power_Zone_1 and Power_Zone_2 (known zone readings)
- OHE Month indicators (sparse vector)

In [15]:
final_assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin",
               "Power_Zone_1", "Power_Zone_2", "Month_ohe"],
    outputCol="features"
)

## 2.7 Assembling the pipeline

Built the full transformation pipeline using all stages defined above. Note: pca_model is the fitted PCA transformer, not the raw PCA estimator.

In [16]:
pipeline = Pipeline(stages=[
    sql_cast,          # 2a : cast Hour to Double
    binarizer,         # 2b : binarize Hour into night/day flag
    ohe,               # 2c : one-hot encode Month
    weather_assembler, # 2d (part 1) : bundle weather cols into a vector
    pca_model,         # 2d (part 2) : apply pre-fitted PCA (2 components)
    sql_label,         # 2e : alias Power_Zone_3 as label
    final_assembler    # 2f : assemble all predictors into features vector
])

## 2.8 Verifying the pipeline

Transformed the data and confirmed features + label columns exist

In [17]:
pipeline_check = pipeline.fit(spark_df).transform(spark_df)
pipeline_check.select("label", "features").show(5, truncate=True)

+-----------+--------------------+
|      label|            features|
+-----------+--------------------+
|20240.96386|(17,[0,1,3,4,6],[...|
|20131.08434|(17,[0,1,3,4,6],[...|
|19668.43373|(17,[0,1,3,4,6],[...|
|18899.27711|(17,[0,1,3,4,6],[...|
|18442.40964|(17,[0,1,3,4,6],[...|
+-----------+--------------------+
only showing top 5 rows


# 3. Cross-Validated Elastic Net

Built a `LinearRegression` estimator, defined the 11×11 parameter grid(121 combinations), wrapped the pipeline inside a `CrossValidator`, and fitted on the full dataset.

Note: 121 combinations × 5 folds = 605 model fits

In [18]:
# Imports for model fitting, hyperparameter search, and evaluation
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

## 3.1 Building the parameter grid

### 3.1.1 Instantiating a LinearRegression estimator 

Elastic net is controlled via regParam + elasticNetParam

In [19]:
lr = LinearRegression()

### 3.1.2 Building an 11x11 grid 

All combinations of regParam and elasticNetParam = 121 total

In [20]:
param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam,        [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .build()
)

print(f"Total parameter combinations: {len(param_grid)}")  # should be 121

Total parameter combinations: 121


## 3.2 Configuring the `CrossValidator`

- Wrapped the full transformation pipeline + LR model in a `CrossValidator` 
- `estimator` = pipeline (all feature engineering + LR in one object) 
- `numFolds=5` with RMSE as the selection criterion; no train/test split per assignment instructions

In [21]:
crossval = CrossValidator(
    estimator=Pipeline(stages=pipeline.getStages() + [lr]),
    estimatorParamMaps=param_grid,
    evaluator=RegressionEvaluator(metricName="rmse"),
    numFolds=5
)

## 3.3 Fitting the `CrossValidator`

On the full dataset (605 model fits total)

In [22]:
cv_model = crossval.fit(spark_df)
print("Cross-validation complete.")

26/04/30 15:40:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/30 15:40:46 WARN Instrumentation: [eddafec7] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 15:40:47 WARN Instrumentation: [eddafec7] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 15:40:48 WARN Instrumentation: [ea7d414f] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 15:40:49 WARN Instrumentation: [ea7d414f] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 15:40:50 WARN Instrumentation: [731e0d76] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 15:40:50 WARN Instrumentation: [731e0d76] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Cross-validation complete.


# 4. Report Results

Extract the best tuning parameters, CV RMSE, and training RMSE from the fitted cross-validator.

## 4.1 Finding optimal tuning parameters

`cv_model.bestModel` is the `PipelineModel` that won the grid search. Its `.stages[-1]` is the fitted `LinearRegression`, and `.getRegParam()` / `.getElasticNetParam()` pull the exact values Spark chose.

### 4.1.1 Pulling the best LinearRegression model 

Located the best model out of the winning pipeline. Pulling the last stage (the fitted LinearRegression)

In [23]:
best_lr = cv_model.bestModel.stages[-1]

### 4.1.2 Extracting the optimal elastic net tuning parameters

In [24]:
best_reg_param        = best_lr.getRegParam()
best_elastic_net_param = best_lr.getElasticNetParam()

print(f"Optimal regParam:        {best_reg_param}")
print(f"Optimal elasticNetParam: {best_elastic_net_param}")

Optimal regParam:        0.1
Optimal elasticNetParam: 0.1


## 4.2 Getting best CV RMSE

`cv_model.avgMetrics` stores a list of average fold-RMSE values 
- one per grid combination and 
- in the same order as the `paramGrid` 

The minimum, given by `min()`, is the CV error for the chosen best model (the winning parameter set).

In [25]:
cv_rmse = min(cv_model.avgMetrics)

print(f"5-fold CV RMSE: {cv_rmse:.4f}")

5-fold CV RMSE: 2148.1004


## 4.3 Getting Training RMSE

### 4.3.1 Using the fitted cv_model on the full training set

Called `.transform(spark_df)` on the cross-validator model to run the winning pipeline (all transformations and best LR) over the original data and produce a `prediction` column 

In [26]:
train_predictions = cv_model.transform(spark_df)

### 4.3.2 Evaluating predictions against the true labels using RMSE

Computed the RMSE of `prediction` against `label` using the `RegressionEvaluator`.

In [27]:
evaluator = RegressionEvaluator(
    labelCol = "label",
    predictionCol = "prediction",
    metricName = "rmse"
)

train_rmse = evaluator.evaluate(train_predictions)
print(f"Training Set RMSE: {train_rmse:.4f}")

Training Set RMSE: 2147.0973


Note: The Training Set RMSE will typically be lower than the CV RMSE since the model has seen this data. 

# 5. Residuals

Computed a `residual` column (`label - prediction`) from the full-data predictions generated in Step 4, then displayed a clean three-column view.

## 5.1 Computing residuals

Added a derived column on top of the same DataFrame since `train_predictions` already exists

- Positive residual = under-predicted
- Negative residual = over-predicted

In [28]:
from pyspark.sql.functions import col

residuals_df = train_predictions.withColumn(
    "residual", col("label") - col("prediction")
)

## 5.2 Displaying residuals

Trimmed output to just the three columns of interest. 

In [29]:
residuals_df.select("label", "prediction", "residual").show()

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20879.64826490436|-638.6844049043611|
|20131.08434| 18659.94760145423|1471.1367385457706|
|19668.43373|18204.475225977243|1463.9585040227576|
|18899.27711|  17590.4074787644|1308.8696312356005|
|18442.40964|16997.066429850194|1445.3432101498074|
|18130.12048|16517.468764606543|1612.6517153934583|
|17945.06024|16093.047989112592|1852.0122508874065|
|17459.27711|15722.506401187877|1736.7707088121224|
|17025.54217|15270.872472377929| 1754.669697622072|
|16794.21687|14938.183526222789|1856.0333437772115|
|16638.07229|14652.279057768608|1985.7932322313918|
|16395.18072|14414.804017338776| 1980.376702661224|
|16117.59036|14082.745977971214|2034.8443820287866|
| 15822.6506|13624.760839394568|2197.8897606054325|
|15672.28916|13450.252372554589|2222.0367874454114|
|15597.10843| 13302.20160902586|  2294.90682097414|
|15510.36145

# Part 2: Streaming Part

# 1. Setup

Created the watch folder where our producer script will drop CSV files.
Spark's readStream will monitor this directory for incoming data.

## 1.1 Creating watch folder

In [30]:
import os

watch_folder = "stream_csv"

os.makedirs(watch_folder, exist_ok=True)

# exist_ok=True means the cell won't throw an error if re-run

print(f"Watch folder '{watch_folder}' is ready at: {os.path.abspath(watch_folder)}")

Watch folder 'stream_csv' is ready at: /home/jupyter-chhammet@ncsu.edu/ST554_Final/stream_csv


## 1.2 Adding a .gitkeep 

so Git tracks the otherwise-empty folder

In [31]:
open(os.path.join(watch_folder, ".gitkeep"), "w").close()

# 2. Stream Schema

Reused the schema from the static Spark DataFrame built in Part 1. This ensured the stream reader knows the exact column names and types
expected in each incoming CSV file, rather than inferring (which is not supported in structured streaming).

## 2.1 Defining the stream schema

Spark structured streaming requires a schema up front. It cannot infer one like a regular spark.read call does.

In [32]:
stream_schema = spark_df.schema

## 2.2 Confirming schema looks correct

In [33]:
print(stream_schema)

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])


## 2.3 Set up `readStream`

Pointed Spark's streaming reader to the watch folder (`stream_csv`)
- `.schema` : supplied the schema defined above
- `.option("header")` : set to True, telling spark to skip the header row

In [34]:
stream_df = (spark.readStream
               .schema(stream_schema)
               .option("header", "true")
               .csv("stream_csv/"))

# 3. Stream Transformation and Joining

Two separate transformations are applied to the same `stream_df` object. Spark's structured streaming allows a single stream to be referenced multiple times. Each reference just describes a logical transformation plan, so no data is duplicated or consumed twice.

## 3.1 Applying the fitted cross-validated model

Used the fitted cross-validation model as a transformer on the incoming stream. `cvModel` was trained in Part 1 and its full pipeline (transformations + linear regression) is reused here. This way, every batch of arriving data rows goes through the same feature-engineering steps before prediction

In [35]:
from pyspark.sql.functions import col

pred_df = cv_model.transform(stream_df)

pred_df = pred_df.withColumn("residual", col("label") - col("prediction"))

pred_df = pred_df.select("label", "prediction", "residual")

## 3.2 Transforming a second time on same `stream_df`

Renamed `Power_Zone_3` to `label` so this stream shares the join key with `pred_df` above. All other columns were left the same for context after the join. 

In [36]:
label_df = stream_df.withColumn("label", col("Power_Zone_3"))

## 3.3 Joining streams

Joined `pred_df` and `label_df` on the shared `label` column. 
- `pred_df` has (label, prediction, residual)
- `label_df` has the full original row with renamed `label` column

Both originated from the same underlying stream so the joins can be done within each micro-batch. The join brings them together so the output shows predictions alongside the raw incoming features

In [37]:
joined_df = pred_df.join(label_df, on="label")

# 4. Write Stream

Output each incoming micro-batch to the console

## 4.1 Configuring `writeStream`

- `.format("console")` : Prints out each batch to stout
- `.outputMode("append")` : Only new rows added since last trigger. No running aggregations updating previously written results.
- `.start())` : Kicks off the streaming query, returns a StreamingQuery object

In [39]:
query = (joined_df
         .writeStream
         .format("console")
         .outputMode("append")
         .start())

26/04/30 15:52:47 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-fe3252d8-5481-4dfb-af42-0a5ef53316dd. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/30 15:52:47 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
| 22981.7004| 23511.17207739532| -529.4716773953187|       18.5|    82.1|     4.923|                0.051|        0.122| 38022.29508| 22729.41176|  22981.7004|    5|   2|
|40139.43574| 33316.87580380582|  6822.559936194179|      26.06|   46.45|     4.906|                57.73|        67.62| 49314.80577| 33795.14256| 40139.43574|    8|  19|
|13943.13253|13132.324557707225|   810.807972292

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|15885.59755|16543.149701415576| -657.5521514155753|      24.98|   44.61|     0.272|                237.4|        53.66| 36643.53982| 22617.87942| 15885.59755|    9|  18|
|9968.787515| 11883.94584427931|-1915.1583292793093|      17.54|   45.91|     0.075|                258.0|         77.3| 31537.64259| 26330.77631| 9968.787515|   12|  16|
|16249.69988|17874.661302599707|-1624.9614225997

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|15885.59755|16543.149701415576| -657.5521514155753|      24.98|   44.61|     0.272|                237.4|        53.66| 36643.53982| 22617.87942| 15885.59755|    9|  18|
|9968.787515| 11883.94584427931|-1915.1583292793093|      17.54|   45.91|     0.075|                258.0|         77.3| 31537.64259| 26330.77631| 9968.787515|   12|  16|
|16249.69988|17874.661302599707|-1624.9614225997

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|13251.49798| 13335.55153408426|  -84.0535540842593|      16.79|    77.9|      4.92|                0.062|         0.13| 22687.47541| 14474.30341| 13251.49798|    5|   4|
|24416.49231|25455.050563741686| -1038.558253741685|      18.88|   58.31|     0.083|                0.084|        0.111| 38692.45033| 24414.13721| 24416.49231|    6|   0|
|19224.07524|21905.134694708322|-2681.0594547083

...

-------------------------------------------
Batch: 4
-------------------------------------------


+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|18183.64372| 18658.03061778158|-474.38689778158005|      21.75|   63.16|     4.917|                842.0|        52.92| 36284.85246| 22246.43963| 18183.64372|    5|  14|
|15215.42169|19468.662137540246| -4253.240447540247|      17.04|    74.1|     0.076|                444.5|        116.9| 35702.27848| 22220.06079| 15215.42169|    1|  11|
| 18754.6988|12260.170603565903|  6494.528196434096|       19.8|   67.35|     4.926|                429.5|         81.9| 30252.30769| 22009.09091

-------------------------------------------
Batch: 5
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|24203.57789|23430.211608677797|  773.3662813222036|       8.74|    86.9|     0.083|                0.062|        0.178| 40063.72881| 24258.96657| 24203.57789|    2|  21|
|30324.35146| 28681.54242197446| 1642.8090380255417|      32.28|   39.45|     4.922|                515.1|        66.23| 37998.13953| 25537.97468| 30324.35146|    7|  17|
|17874.65587|17149.895361709277|  724.7605082907

-------------------------------------------
Batch: 5
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|24203.57789|23430.211608677797|  773.3662813222036|       8.74|    86.9|     0.083|                0.062|        0.178| 40063.72881| 24258.96657| 24203.57789|    2|  21|
|17198.70968|16073.958362789495| 1124.7513172105046|      25.41|   42.99|     0.082|                803.0|        480.2| 34388.42553| 19928.04878| 17198.70968|    3|  14|
|30324.35146| 28681.54242197446| 1642.8090380255

-------------------------------------------
Batch: 6
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|17198.70968|16073.958362789495| 1124.7513172105046|      25.41|   42.99|     0.082|                803.0|        480.2| 34388.42553| 19928.04878| 17198.70968|    3|  14|
|18699.63636| 17722.71061754378|  976.9257424562202|      24.72|   39.91|     4.922|                417.8|        454.4| 33022.34661| 19055.80448| 18699.63636|    4|  17|
| 19404.6395| 24808.75750598505|-5404.1180059850